In [11]:
# Imports used throughout the notebook.
# `glob`/`os` locate the source .xlsx files (and skip Office lock files),
# `pandas` does all the data loading/analysis below.
import glob
import os

import pandas as pd


# ACSEL Data Exploration

Explore the structure of every file under `ACSEL/`:
- The 9 `GPContest_Grade_{4,5,6}_{year}.xlsx` assessment files (each with `Assessment Data`, `Competency Mapping`, and `District and GP Mapping` sheets)
- `district_crosswalk.xlsx` (district name crosswalk + dataset sources)
- `Note on Data.docx` (background notes on the datathon datasets)

### Sheet structure per file

Find every `GPContest_*.xlsx` file (ignoring `~$...` Office lock/temp files)
and print each sheet's shape and column names, so we know what we're working
with before writing any real analysis code.

In [12]:
xlsx_files = sorted(
    f
    for f in glob.glob("ACSEL/Akshara_Data_For Datathon/**/*.xlsx", recursive=True)
    if not os.path.basename(f).startswith("~$")
)

for f in xlsx_files:
    xls = pd.ExcelFile(f)
    print("=====", f, "=====")
    print("Sheets:", xls.sheet_names)
    for s in xls.sheet_names:
        if s == "Competency Mapping":
            raw = xls.parse(s, header=None)
            header_mask = raw.apply(lambda row: row.astype(str).str.strip().eq("Questions").any(), axis=1)
            header_row_idx = header_mask.idxmax()
            full = raw.iloc[header_row_idx + 1 :]
            full.columns = raw.iloc[header_row_idx]
            full = full.loc[:, full.columns.notna()]  # drop leading blank column, if any
        else:
            full = xls.parse(s)
        print(f"--sheet: {s} shape={full.shape}")
        print("columns:", list(full.columns))
    print()

===== ACSEL/Akshara_Data_For Datathon/2022-23/GPContest_Grade_4_2022-23.xlsx =====
Sheets: ['Assessment Data', 'Competency Mapping', 'District and GP Mapping']
--sheet: Assessment Data shape=(104015, 28)
columns: ['State', 'District', 'Block', 'Cluster', 'GP Name', 'GP ID', 'Gender', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Unique Identifier']
--sheet: Competency Mapping shape=(20, 3)
columns: ['Questions', 'Question Name', 'Competency']
--sheet: District and GP Mapping shape=(12635, 6)
columns: ['State', 'District', 'Block', 'Cluster', 'GP Name', 'GP ID']

===== ACSEL/Akshara_Data_For Datathon/2022-23/GPContest_Grade_5_2022-23.xlsx =====
Sheets: ['Assessment Data', 'Competency Mapping', 'District and GP Mapping']
--sheet: Assessment Data shape=(113164, 28)
columns: ['State', 'District', 'Block', 'Cluster', 'GP Name', 'GP ID', 'Gender', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10',

## Assessment Data — pandas analysis

Load every `Assessment Data` sheet into one combined DataFrame (tagging each
row with its `grade` and `year`), then use pandas to summarize scores,
gender/district breakdowns, and per-competency accuracy (via the
`Competency Mapping` sheet).

In [4]:
import re

question_cols = [f"Q{i}" for i in range(1, 21)]


def parse_competency_mapping(xls):
    raw = xls.parse("Competency Mapping", header=None)
    header_mask = raw.apply(lambda row: row.astype(str).str.strip().eq("Questions").any(), axis=1)
    header_row_idx = header_mask.idxmax()
    header_row = raw.iloc[header_row_idx]
    col_idx = {str(val).strip(): col for col, val in header_row.items() if pd.notna(val)}

    comp = raw.iloc[header_row_idx + 1 :][[col_idx["Questions"], col_idx["Question Name"], col_idx["Competency"]]]
    comp.columns = ["Question", "Question Name", "Competency"]
    return comp.dropna(subset=["Question"])


assessment_frames = []
competency_frames = []
for f in xlsx_files:
    match = re.search(r"Grade_(\d+)_(\d{4}-\d{2})", f)
    grade, year = match.group(1), match.group(2)

    xls = pd.ExcelFile(f)

    df = xls.parse("Assessment Data")
    df["grade"] = int(grade)
    df["year"] = year
    assessment_frames.append(df)

    comp = parse_competency_mapping(xls)
    comp["grade"] = int(grade)
    comp["year"] = year
    competency_frames.append(comp)

assessment_df = pd.concat(assessment_frames, ignore_index=True)
competency_df = pd.concat(competency_frames, ignore_index=True)

assessment_df["score"] = assessment_df[question_cols].sum(axis=1)
assessment_df["pct_correct"] = assessment_df["score"] / len(question_cols)

print("Combined assessment rows:", assessment_df.shape)
print("Combined competency rows:", competency_df.shape)
assessment_df.head()

Combined assessment rows: (1379087, 32)
Combined competency rows: (180, 5)


,State,District,Block,Cluster,GP Name,GP ID,Gender,Q1,Q2,Q3,...,Q16,Q17,Q18,Q19,Q20,Unique Identifier,grade,year,score,pct_correct
0,Karnataka,belagavi,bailhongal,ambadgatti,ambadgatti,648,female,1,1,1,...,1,1,1,1,0,Grade_4_22_23_1,4,2022-23,16,0.80
1,Karnataka,belagavi,bailhongal,ambadgatti,ambadgatti,648,female,0,1,1,...,0,0,0,1,0,Grade_4_22_23_2,4,2022-23,11,0.55
2,Karnataka,belagavi,bailhongal,ambadgatti,ambadgatti,648,female,1,0,1,...,1,1,0,1,1,Grade_4_22_23_3,4,2022-23,10,0.50
3,Karnataka,belagavi,bailhongal,ambadgatti,ambadgatti,648,female,1,1,1,...,1,1,1,1,1,Grade_4_22_23_4,4,2022-23,18,0.90
4,Karnataka,belagavi,bailhongal,ambadgatti,ambadgatti,648,male,1,1,1,...,1,0,1,0,0,Grade_4_22_23_5,4,2022-23,9,0.45


### Score distribution by year and grade

Summary stats (`count`, `mean`, `std`, quartiles, etc.) of `pct_correct` for
each `(year, grade)` group — a quick check for how overall performance
trends across the three school years and grades.

In [5]:
assessment_df.groupby(["year", "grade"])["pct_correct"].describe()

count      mean       std  min   25%   50%   75%  max
year    grade                                                          
2022-23 4      104015.0  0.496059  0.300275  0.0  0.25  0.50  0.75  1.0
        5      113164.0  0.561462  0.283786  0.0  0.35  0.60  0.80  1.0
        6       95523.0  0.625107  0.265296  0.0  0.45  0.70  0.85  1.0
2023-24 4      156940.0  0.477971  0.282846  0.0  0.25  0.50  0.70  1.0
        5      161979.0  0.513667  0.275268  0.0  0.30  0.50  0.75  1.0
        6      151938.0  0.545155  0.280879  0.0  0.30  0.55  0.80  1.0
2024-25 4      209867.0  0.578241  0.298456  0.0  0.35  0.65  0.85  1.0
        5      204008.0  0.517045  0.279855  0.0  0.30  0.55  0.75  1.0
        6      181653.0  0.492808  0.283105  0.0  0.25  0.50  0.75  1.0

### Trend plot: score distribution by year, per grade

Turn the `describe()` summary above into a visual trend: for each grade, plot
mean `pct_correct` across the three school years as a line, with a shaded
band showing the 25th-75th percentile range (IQR) so we can see both the
central trend and the spread of scores each year.


In [14]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Per (year, grade) summary stats to drive the trend + IQR band.
trend_stats = (
    assessment_df.groupby(["grade", "year"])["pct_correct"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .reset_index()
    .sort_values(["grade", "year"])
)

grades = sorted(trend_stats["grade"].unique())
grade_colors = {4: "#636EFA", 5: "#EF553B", 6: "#00CC96"}


y_min = (trend_stats["25%"].min() - 0.03)
y_max = (trend_stats["75%"].max() + 0.03)

fig = make_subplots(
    rows=1,
    cols=len(grades),
    subplot_titles=[f"Grade {g}" for g in grades],
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

for i, grade_val in enumerate(grades, start=1):
    g = trend_stats[trend_stats["grade"] == grade_val]
    color = grade_colors.get(grade_val, "#AB63FA")
    rgba = "rgba({}, {}, {}, 0.20)".format(
        int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    )

    # Shaded IQR band (25th-75th percentile), drawn first so the mean line sits on top.
    fig.add_trace(
        go.Scatter(
            x=list(g["year"]) + list(g["year"])[::-1],
            y=list(g["75%"]) + list(g["25%"])[::-1],
            fill="toself",
            fillcolor=rgba,
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=i,
    )

    # Mean trend line with markers + on-chart value labels.
    fig.add_trace(
        go.Scatter(
            x=g["year"],
            y=g["mean"],
            mode="lines+markers+text",
            name=f"Grade {grade_val}",
            line=dict(color=color, width=3),
            marker=dict(size=10, line=dict(color="white", width=1.5)),
            text=[f"{v:.1%}" for v in g["mean"]],
            textposition="top center",
            textfont=dict(size=12, color=color),
            hovertemplate="Year: %{x}<br>Mean: %{y:.1%}<br>IQR: %{customdata[0]:.1%}-%{customdata[1]:.1%}<extra></extra>",
            customdata=g[["25%", "75%"]].values,
            showlegend=False,
        ),
        row=1,
        col=i,
    )

    fig.update_xaxes(title_text="School year", row=1, col=i)

fig.update_yaxes(
    title_text="Percent correct",
    tickformat=".0%",
    range=[y_min, y_max],
    gridcolor="rgba(0,0,0,0.08)",
    row=1,
    col=1,
)
for i in range(2, len(grades) + 1):
    fig.update_yaxes(range=[y_min, y_max], gridcolor="rgba(0,0,0,0.08)", row=1, col=i)

fig.update_layout(
    title=dict(
        text="Score trend by grade across school years<br><sup>Line + labels = mean pct_correct, shaded band = 25th-75th percentile (same y-scale across panels)</sup>",
        x=0.02,
        xanchor="left",
    ),
    template="plotly_white",
    font=dict(size=13),
    width=1050,
    height=480,
    margin=dict(t=100),
)
fig.show()


### Gender gap in scores

Mean `pct_correct` split by `Gender`, for each `(year, grade)` group, to see
whether there's a consistent gap between female and male students.

In [6]:
# Gender breakdown of average score
assessment_df.groupby(["year", "grade", "Gender"])["pct_correct"].mean().unstack("Gender")

Gender           female      male
year    grade                    
2022-23 4      0.505665  0.484873
        5      0.573419  0.547839
        6      0.632878  0.616915
2023-24 4      0.485661  0.468892
        5      0.521311  0.504650
        6      0.559237  0.529031
2024-25 4      0.590398  0.564468
        5      0.523983  0.508854
        6      0.503832  0.480280

### Trend plot: gender gap by year, per grade

Same small-multiples style as the score trend above: one panel per grade,
with a line per gender showing mean `pct_correct` across school years and
value labels, on a shared y-axis so the gap is easy to compare across
grades.


In [16]:
gender_stats = (
    assessment_df.groupby(["grade", "year", "Gender"])["pct_correct"]
    .mean()
    .reset_index()
    .sort_values(["grade", "Gender", "year"])
)

gender_colors = {"female": "#EF553B", "male": "#636EFA"}
genders = sorted(gender_stats["Gender"].dropna().unique())

y_min_g = gender_stats["pct_correct"].min() - 0.03
y_max_g = gender_stats["pct_correct"].max() + 0.03

fig = make_subplots(
    rows=1,
    cols=len(grades),
    subplot_titles=[f"Grade {g}" for g in grades],
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

for i, grade_val in enumerate(grades, start=1):
    for gender in genders:
        gg = gender_stats[(gender_stats["grade"] == grade_val) & (gender_stats["Gender"] == gender)]
        color = gender_colors.get(gender, "#AB63FA")

        fig.add_trace(
            go.Scatter(
                x=gg["year"],
                y=gg["pct_correct"],
                mode="lines+markers+text",
                name=gender,
                legendgroup=gender,
                showlegend=(i == 1),
                line=dict(color=color, width=3),
                marker=dict(size=10, line=dict(color="white", width=1.5)),
                text=[f"{v:.1%}" for v in gg["pct_correct"]],
                textposition="top center",
                textfont=dict(size=12, color=color),
                hovertemplate=f"{gender}<br>" + "Year: %{x}<br>Mean: %{y:.1%}<extra></extra>",
            ),
            row=1,
            col=i,
        )

    fig.update_xaxes(title_text="School year", row=1, col=i)

fig.update_yaxes(
    title_text="Percent correct",
    tickformat=".0%",
    range=[y_min_g, y_max_g],
    gridcolor="rgba(0,0,0,0.08)",
    row=1,
    col=1,
)
for i in range(2, len(grades) + 1):
    fig.update_yaxes(range=[y_min_g, y_max_g], gridcolor="rgba(0,0,0,0.08)", row=1, col=i)

fig.update_layout(
    title=dict(
        text="Gender gap trend by grade across school years<br><sup>Line + labels = mean pct_correct per gender (same y-scale across panels)</sup>",
        x=0.02,
        xanchor="left",
    ),
    template="plotly_white",
    font=dict(size=13),
    width=1050,
    height=480,
    margin=dict(t=100),
    legend_title="Gender",
)
fig.show()


### District performance (all years and grades)

Rank districts by mean `pct_correct` across every `(year, grade)` combination:
1. A heatmap of mean score per district × year, with one panel per grade,
   districts sorted by their overall average score (best to worst) so
   patterns are easy to scan.
2. An overall top/bottom 10 table (averaged across all years and grades,
   with total student `count` for context).

In [17]:
# District x year mean score, one heatmap panel per grade, districts sorted
# by their overall (all years/grades) average score.
district_overall = assessment_df.groupby("District")["pct_correct"].mean().sort_values(ascending=False)
district_order = district_overall.index.tolist()

district_year_grade = (
    assessment_df.groupby(["grade", "District", "year"])["pct_correct"]
    .mean()
    .reset_index()
)
years = sorted(district_year_grade["year"].unique())

fig = make_subplots(
    rows=1,
    cols=len(grades),
    subplot_titles=[f"Grade {g}" for g in grades],
    horizontal_spacing=0.08,
)

for i, grade_val in enumerate(grades, start=1):
    pivot = (
        district_year_grade[district_year_grade["grade"] == grade_val]
        .pivot(index="District", columns="year", values="pct_correct")
        .reindex(district_order)[years]
    )
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=pivot.index,
            coloraxis="coloraxis",
            hovertemplate="District: %{y}<br>Year: %{x}<br>Mean: %{z:.1%}<extra></extra>",
        ),
        row=1,
        col=i,
    )

fig.update_layout(
    title=dict(
        text="District performance heatmap by grade<br><sup>Rows = districts (sorted best to worst overall), columns = school year, color = mean pct_correct</sup>",
        x=0.02,
        xanchor="left",
    ),
    coloraxis=dict(colorscale="RdYlGn", colorbar=dict(title="Mean", tickformat=".0%")),
    template="plotly_white",
    font=dict(size=11),
    width=1100,
    height=900,
    margin=dict(t=100),
)
fig.update_yaxes(autorange="reversed", row=1, col=1)
for i in range(2, len(grades) + 1):
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=i)
fig.show()

# Overall top/bottom 10 districts, averaged across all years and grades.
district_scores = assessment_df.groupby("District")["pct_correct"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print("Top 10 districts (all years/grades):")
display(district_scores.head(10))
print("Bottom 10 districts (all years/grades):")
display(district_scores.tail(10))

Top 10 districts (all years/grades):


,mean,count
District,,
udupi,0.742700,2815
dakshina kannada,0.729205,1623
uttara kannada sirsi,0.699625,801
tumakuru,0.667008,52084
belagavi,0.656637,56630
dharwad,0.629222,66812
bengaluru rural,0.615830,27631
belagavi chikkodi,0.611315,59062
chikkamagaluru,0.600777,28894


Bottom 10 districts (all years/grades):


,mean,count
District,,
davanagere,0.492885,38336
chikkaballapura,0.486861,36469
yadagiri,0.482332,44853
koppal,0.473977,63307
raichur,0.465686,70124
bidar,0.454034,42933
vijayanagar,0.432556,61409
chitradurga,0.424103,87338
ballari,0.422798,46061


### Competency-level accuracy (overall, and by grade/year)

Each question (`Q1`-`Q20`) maps to a `Competency` (e.g. "number sense",
"place value") via the per-file `Competency Mapping` sheet, and that mapping
differs by grade/year. To compute accuracy per competency we:
1. `melt` the wide `Q1..Q20` columns into a long `(grade, year, Question, correct)` table.
2. Join that with `competency_df` on `(Question, grade, year)` to attach the right competency for each row's grade/year.
3. Average `correct` per `Competency`, both overall and split by `(grade, year)`, so we can see the ranking overall as well as how it shifts across grades and years.

In [20]:
# Per-competency accuracy: melt question columns to long format, join with the
# (grade, year)-specific Competency Mapping, then average correctness by competency.
long_df = assessment_df.melt(
    id_vars=["grade", "year"],
    value_vars=question_cols,
    var_name="Question",
    value_name="correct",
)

competency_lookup = competency_df.rename(columns={"Question": "Question"})
merged = long_df.merge(competency_lookup, on=["Question", "grade", "year"], how="left")

overall_competency = merged.groupby("Competency")["correct"].mean().sort_values(ascending=False)
print("Overall competency accuracy (all grades/years combined):")
display(overall_competency)

# Accuracy per competency, split by grade and year, ordered by overall accuracy.
competency_by_grade_year = (
    merged.groupby(["Competency", "grade", "year"])["correct"]
    .mean()
    .reset_index()
)
competency_order = overall_competency.index.tolist()

fig = make_subplots(
    rows=1,
    cols=len(grades),
    subplot_titles=[f"Grade {g}" for g in grades],
    horizontal_spacing=0.08,
)

for i, grade_val in enumerate(grades, start=1):
    pivot_comp = (
        competency_by_grade_year[competency_by_grade_year["grade"] == grade_val]
        .pivot(index="Competency", columns="year", values="correct")
        .reindex(competency_order)[years]
    )
    fig.add_trace(
        go.Heatmap(
            z=pivot_comp.values,
            x=pivot_comp.columns,
            y=pivot_comp.index,
            coloraxis="coloraxis",
            hovertemplate="Competency: %{y}<br>Year: %{x}<br>Accuracy: %{z:.1%}<extra></extra>",
        ),
        row=1,
        col=i,
    )

fig.update_layout(
    title=dict(
        text="Competency accuracy by grade and year<br><sup>Rows = competencies (sorted best to worst overall), columns = school year, color = mean accuracy</sup>",
        x=0.02,
        xanchor="left",
    ),
    coloraxis=dict(colorscale="RdYlGn", colorbar=dict(title="Accuracy", tickformat=".0%")),
    template="plotly_white",
    font=dict(size=11),
    width=1100,
    height=500,
    margin=dict(t=100),
)
fig.update_yaxes(autorange="reversed", row=1, col=1)
for i in range(2, len(grades) + 1):
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=i)
fig.show()

Overall competency accuracy (all grades/years combined):


Competency
fraction          0.634348
addition          0.615956
number sense      0.593427
data handling     0.593322
mensuration       0.555680
shapes            0.549159
place value       0.537588
measurement       0.507689
subtraction       0.496643
multiplication    0.484944
division          0.436058
Name: correct, dtype: float64

In [21]:
# Quick check: does each GP belong to exactly one Cluster, each Cluster to
# exactly one Block, each Block to exactly one District? Confirms the
# nesting order of the geographic hierarchy used in the data.
hier_cols = ["State", "District", "Block", "Cluster", "GP Name", "GP ID"]
hier = assessment_df[hier_cols].drop_duplicates()

print("Unique States:", hier["State"].nunique())
print("Unique Districts:", hier["District"].nunique())
print("Unique Blocks:", hier["Block"].nunique())
print("Unique Clusters:", hier["Cluster"].nunique())
print("Unique GP Names:", hier["GP Name"].nunique())
print("Unique GP IDs:", hier["GP ID"].nunique())
print()

# Does every Block map to exactly one District? Every Cluster to one Block? Every GP ID to one Cluster?
print("Blocks with >1 District:", hier.groupby("Block")["District"].nunique().gt(1).sum())
print("Clusters with >1 Block:", hier.groupby("Cluster")["Block"].nunique().gt(1).sum())
print("GP IDs with >1 Cluster:", hier.groupby("GP ID")["Cluster"].nunique().gt(1).sum())

Unique States: 1
Unique Districts: 31
Unique Blocks: 166
Unique Clusters: 2896
Unique GP Names: 4631
Unique GP IDs: 5034

Blocks with >1 District: 2
Clusters with >1 Block: 181
GP IDs with >1 Cluster: 1718


### Competency-level accuracy at the Gram Panchayat (GP) level

Same competency join as before, but grouped down to the GP level using the
composite key `(District, Block, Cluster, GP Name, GP ID)` — not `GP ID`
alone — since GP IDs/names aren't globally unique (see the hierarchy check
above). Small GPs are noisy, so we require a minimum number of responses
before ranking a GP, and separately surface each GP's single weakest
competency (useful for targeting interventions).


In [24]:
gp_key_cols = ["District", "Block", "Cluster", "GP Name", "GP ID"]

# Melt again, this time keeping the GP hierarchy columns so we can group by
# the composite GP key instead of just grade/year.
gp_long_df = assessment_df.melt(
    id_vars=gp_key_cols + ["grade", "year"],
    value_vars=question_cols,
    var_name="Question",
    value_name="correct",
)
gp_merged = gp_long_df.merge(competency_lookup, on=["Question", "grade", "year"], how="left")

MIN_RESPONSES = 200  # minimum (student x question) responses required to rank a GP

# Overall accuracy per GP (all competencies/grades/years combined), filtered
# to GPs with enough data to be a reliable estimate.
gp_overall = (
    gp_merged.groupby(gp_key_cols)["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy"})
)

# Number of students per GP (each assessment_df row = one student sitting),
# joined in so the table shows students, not just exploded question counts.
gp_n_students = assessment_df.groupby(gp_key_cols).size().rename("n_students")
gp_overall = gp_overall.join(gp_n_students)

gp_overall_reliable = gp_overall[gp_overall["count"] >= MIN_RESPONSES].sort_values("accuracy", ascending=False)

print(f"Total distinct GPs: {len(gp_overall)}")
print(f"GPs with >= {MIN_RESPONSES} responses: {len(gp_overall_reliable)}")
print()
print("Top 15 GPs by overall accuracy:")
display(gp_overall_reliable.head(15))
print("Bottom 15 GPs by overall accuracy:")
display(gp_overall_reliable.tail(15))

# Each GP's single weakest competency (lowest accuracy), for reliable GPs only.
gp_competency = (
    gp_merged.groupby(gp_key_cols + ["Competency"])["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy"})
    .reset_index()
)
gp_competency_reliable = gp_competency.merge(
    gp_overall_reliable.reset_index()[gp_key_cols], on=gp_key_cols, how="inner"
)
weakest_competency_per_gp = gp_competency_reliable.loc[
    gp_competency_reliable.groupby(gp_key_cols)["accuracy"].idxmin()
].set_index(gp_key_cols)

print("\nMost common 'weakest competency' across all reliable GPs:")
display(weakest_competency_per_gp["Competency"].value_counts())


Total distinct GPs: 7190
GPs with >= 200 responses: 6643

Top 15 GPs by overall accuracy:


accuracy  \
District       Block           Cluster          GP Name      GP ID                 
tumakuru       gubbi           bidare           irakasandra  2964       0.968421   
                               chelur           nallur       4698       0.963208   
                               mara shettyhalli tyagatuuru   5844       0.944737   
hassan         channarayapatna urdhu cr patna   nuggehally   4887       0.942308   
tumakuru       gubbi           c kunnala        peddanahalli 4984       0.940909   
bagalkot       bagalkot        urdu east        bhagavati    1260       0.940476   
chikkamagaluru koppa           koppa gramantara koppa rural  3761       0.936957   
tumakuru       gubbi           gubbi hosahalli  kodigehalli  213009348  0.930952   
udupi          byndoor         navunda          nada         4626       0.930556   
yadagiri       shorapur        rukmapur         hemnoor      2578       0.928571   
tumakuru       kunigal         urdu 2 (rural)   n m pura     4617       0.925000   
kalaburgi      aland           aland south urdu hadalgi      2290       0.922727   
bagalkot       bagalkot        nainegali        hosur        6503       0.919444   
                                                             213009604  0.915789   
ramanagara     ramanagara      sugganahalli     sugganahalli 5516       0.914091   

                                                                        count  \
District       Block           Cluster          GP Name      GP ID              
tumakuru       gubbi           bidare           irakasandra  2964         380   
                               chelur           nallur       4698        1060   
                               mara shettyhalli tyagatuuru   5844         380   
hassan         channarayapatna urdhu cr patna   nuggehally   4887         260   
tumakuru       gubbi           c kunnala        peddanahalli 4984         220   
bagalkot       bagalkot        urdu east        bhagavati    1260         420   
chikkamagaluru koppa           koppa gramantara koppa rural  3761         460   
tumakuru       gubbi           gubbi hosahalli  kodigehalli  213009348    420   
udupi          byndoor         navunda          nada         4626         360   
yadagiri       shorapur        rukmapur         hemnoor      2578         280   
tumakuru       kunigal         urdu 2 (rural)   n m pura     4617         320   
kalaburgi      aland           aland south urdu hadalgi      2290         220   
bagalkot       bagalkot        nainegali        hosur        6503         360   
                                                             213009604    380   
ramanagara     ramanagara      sugganahalli     sugganahalli 5516        2200   

                                                                        n_students  
District       Block           Cluster          GP Name      GP ID                  
tumakuru       gubbi           bidare           irakasandra  2964               19  
                               chelur           nallur       4698               53  
                               mara shettyhalli tyagatuuru   5844               19  
hassan         channarayapatna urdhu cr patna   nuggehally   4887               13  
tumakuru       gubbi           c kunnala        peddanahalli 4984               11  
bagalkot       bagalkot        urdu east        bhagavati    1260               21  
chikkamagaluru koppa           koppa gramantara koppa rural  3761               23  
tumakuru       gubbi           gubbi hosahalli  kodigehalli  213009348          21  
udupi          byndoor         navunda          nada         4626               18  
yadagiri       shorapur        rukmapur         hemnoor      2578               14  
tumakuru       kunigal         urdu 2 (rural)   n m pura     4617               16  
kalaburgi      aland           aland south urdu hadalgi      2290               11  
bagalkot       bagalkot        nainegali        hosur        6503          

Bottom 15 GPs by overall accuracy:


,,,,,accuracy,count,n_students
District,Block,Cluster,GP Name,GP ID,,,
raichur,manvi,hirekotnekal,byagavat,1431,0.174324,740,37
kalaburgi,aland,bhusanoor,jidga,3071,0.170000,200,10
bidar,bidar,rekulgi urdu,bagdal,921,0.168519,2700,135
ballari,sandur,gollalingammanahalli,bommagatta,1384,0.159375,320,16
vijayanagar,harapanahalli,urdu-harapanahalli,kunchooru,3884,0.155556,540,27
kalaburgi,sedam,sedam urdu,telkur,5662,0.154545,220,11
vijayanagar,hadagali,sovena halli,kombali,3718,0.153846,260,13
chitradurga,molakalmur,nagasamudra,hanagal,2382,0.152083,2400,120
kalaburgi,chittapur,wadi urdu,nalawar,4688,0.150000,300,15



Most common 'weakest competency' across all reliable GPs:


Competency
division          3246
mensuration       1212
subtraction        535
multiplication     458
measurement        359
place value        330
data handling      260
shapes             121
fraction            93
addition            19
number sense        10
Name: count, dtype: int64

In [23]:

# Student counts per GP: each row of assessment_df is one student's single
# assessment sitting (in a given grade/year), so counting rows per GP gives
# the number of students (not question-responses).
gp_student_counts = assessment_df.groupby(gp_key_cols).size().rename("n_students")

print("Total distinct GPs:", len(gp_student_counts))
print("GPs with < 10 students:", (gp_student_counts < 10).sum())
print("GPs with < 10 students (%):", f"{(gp_student_counts < 10).mean():.1%}")


Total distinct GPs: 7190
GPs with < 10 students: 547
GPs with < 10 students (%): 7.6%


### GP-level accuracy split by grade

Same composite GP key filter idea as above, but now grouped by `(GP, grade)`
instead of pooling all grades together — a GP can be strong in one grade and
weak in another, and this is masked when grades are combined.

Rather than guessing a minimum-student threshold, look at the actual
distribution of students-per-GP-per-grade and pick a data-driven cutoff (the
25th percentile), so the filter excludes only the sparsest quarter of GPs
instead of an arbitrary round number.


In [26]:
# Distribution of student counts per (GP, grade), to pick a data-driven
# minimum threshold instead of guessing one.
gp_n_students_by_grade = assessment_df.groupby(gp_key_cols + ["grade"]).size().rename("n_students")

print("Distribution of students per (GP, grade):")
display(gp_n_students_by_grade.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

# Use the 25th percentile of students-per-GP-per-grade as the minimum: this
# excludes only the sparsest quarter of GPs, rather than an arbitrary cutoff.
MIN_STUDENTS_PER_GRADE = int(gp_n_students_by_grade.quantile(0.25))
MIN_RESPONSES_PER_GRADE = MIN_STUDENTS_PER_GRADE * len(question_cols)
print(f"\n25th percentile of students per (GP, grade): {MIN_STUDENTS_PER_GRADE}")
print(f"=> MIN_RESPONSES_PER_GRADE = {MIN_RESPONSES_PER_GRADE} ({MIN_STUDENTS_PER_GRADE} students x {len(question_cols)} questions)")

# Accuracy per GP, per grade (all years combined within each grade).
gp_by_grade = (
    gp_merged.groupby(gp_key_cols + ["grade"])["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy"})
)
gp_by_grade = gp_by_grade.join(gp_n_students_by_grade)

gp_by_grade_reliable = gp_by_grade[gp_by_grade["count"] >= MIN_RESPONSES_PER_GRADE]

for grade_val in grades:
    g = gp_by_grade_reliable.xs(grade_val, level="grade").sort_values("accuracy", ascending=False)
    print(f"===== Grade {grade_val}: {len(g)} GPs with >= {MIN_RESPONSES_PER_GRADE} responses =====")
    print(f"Top 10 GPs (Grade {grade_val}):")
    display(g.head(10))
    print(f"Bottom 10 GPs (Grade {grade_val}):")
    display(g.tail(10))


Distribution of students per (GP, grade):


count    20369.000000
mean        67.705189
std         54.161494
min          1.000000
10%          7.000000
25%         22.000000
50%         56.000000
75%        101.000000
90%        145.000000
max        399.000000
Name: n_students, dtype: float64


25th percentile of students per (GP, grade): 22
=> MIN_RESPONSES_PER_GRADE = 440 (22 students x 20 questions)
===== Grade 4: 5236 GPs with >= 440 responses =====
Top 10 GPs (Grade 4):


accuracy  count  \
District       Block      Cluster       GP Name       GP ID                    
tumakuru       gubbi      chelur        nallur        4698   0.967391    460   
raichur        raichur    gadhar        yeragera      6105   0.950000    460   
ramanagara     ramanagara sugganahalli  sugganahalli  5516   0.933333   1020   
tumakuru       tiptur     hosahalli     dasarighatta  1751   0.932979    940   
haveri         shiggoan   baad          halebankapur  2337   0.929032    620   
chikkamagaluru kadur      chowlahiriyar chowlahiriyur 1686   0.922115   2080   
bagalkot       bagalkot   gulabal       timmapur      5776   0.917857    560   
tumakuru       turuvekere mayasandra    mayasandra    4392   0.917683   1640   
                          muniyur       muniyuru      4564   0.915278   2160   
bagalkot       bagalkot   gulabal       hiregulbal    2630   0.915104   1920   

                                                             n_students  
District       Block      Cluster       GP Name       GP ID              
tumakuru       gubbi      chelur        nallur        4698           23  
raichur        raichur    gadhar        yeragera      6105           23  
ramanagara     ramanagara sugganahalli  sugganahalli  5516           51  
tumakuru       tiptur     hosahalli     dasarighatta  1751           47  
haveri         shiggoan   baad          halebankapur  2337           31  
chikkamagaluru kadur      chowlahiriyar chowlahiriyur 1686          104  
bagalkot       bagalkot   gulabal       timmapur      5776           28  
tumakuru       turuvekere mayasandra    mayasandra    4392           82  
                          muniyur       muniyuru      4564          108  
bagalkot       bagalkot   gulabal       hiregulbal    2630           96

Bottom 10 GPs (Grade 4):


...

accuracy  count  \
District    Block      Cluster      GP Name        GP ID                        
davanagere  jagalur    kechenahalli kechchenahalli 3517       0.198592   1420   
bidar       bidar      manhalli     manahalli      4214       0.193023    860   
                       rekulgi urdu bagdal         921        0.189583    480   
ballari     sandur     chornur      agrahara       213009592  0.188298    940   
bidar       bidar      bagdal       bagdal         921        0.176087   1380   
kalaburgi   chittapur  honguntta    hongunta       2743       0.170874   2060   
            sedam      mudhol       mudhol         4505       0.157500   2400   
chitradurga molakalmur nagasamudra  hanagal        2382       0.145349    860   
vijayanagar kudligi    urdu kudligi moraba         4471       0.143182    440   
raichur     manvi      sirwar east  k.gudadinni    3120       0.105000    600   

                                                              n_students  
District    Block      Cluster      GP Name        GP ID                  
davanagere  jagalur    kechenahalli kechchenahalli 3517               71  
bidar       bidar      manhalli     manahalli      4214               43  
                       rekulgi urdu bagdal         921                24  
ballari     sandur     chornur      agrahara       213009592          47  
bidar       bidar      bagdal       bagdal         921                69  
kalaburgi   chittapur  honguntta    hongunta       2743              103  
            sedam      mudhol       mudhol         4505              120  
chitradurga molakalmur nagasamudra  hanagal        2382               43  
vijayanagar kudligi    urdu kudligi moraba         4471               22  
raichur     manvi      sirwar east  k.gudadinni    3120               30

===== Grade 6: 4945 GPs with >= 440 responses =====
Top 10 GPs (Grade 6):


accuracy  \
District          Block      Cluster           GP Name               GP ID                 
davanagere        channagiri lingadahalli      guddada kumaranahalli 2191       0.952174   
udupi             brahmavara hangarakatte      kodi                  3654       0.939130   
dharwad           dharwad    narendra          yadvad                6056       0.933333   
vijayapura        indi       hirebevanur       gubbewad              213009630  0.928000   
bagalkot          bagalkot   shirur            neeralakeri           4785       0.909877   
ramanagara        ramanagara bilagumba         bilagumba             1338       0.904412   
                             kanchugaranahalli kanchugaranahalli     3358       0.903333   
belagavi chikkodi hukkeri    shahabandar       bassapur              1095       0.901852   
                  chikodi    navalihal         khadaklat             3583       0.899398   
tumakuru          gubbi      hindiskere        hindiskere            2606       0.897297   

                                                                                count  \
District          Block      Cluster           GP Name               GP ID              
davanagere        channagiri lingadahalli      guddada kumaranahalli 2191         460   
udupi             brahmavara hangarakatte      kodi                  3654         460   
dharwad           dharwad    narendra          yadvad                6056         840   
vijayapura        indi       hirebevanur       gubbewad              213009630    500   
bagalkot          bagalkot   shirur            neeralakeri           4785        1620   
ramanagara        ramanagara bilagumba         bilagumba             1338        1360   
                             kanchugaranahalli kanchugaranahalli     3358         900   
belagavi chikkodi hukkeri    shahabandar       bassapur              1095         540   
                  chikodi    navalihal         khadaklat             3583        1660   
tumakuru          gubbi      hindiskere        hindiskere            2606         740   

                                                                                n_students  
District          Block      Cluster           GP Name               GP ID                  
davanagere        channagiri lingadahalli      guddada kumaranahalli 2191               23  
udupi             brahmavara hangarakatte      kodi                  3654               23  
dharwad           dharwad    narendra          yadvad                6056               42  
vijayapura        indi       hirebevanur       gubbewad              213009630          25  
bagalkot          bagalkot   shirur            neeralakeri           4785               81  
ramanagara        ramanagara bilagumba         bilagumba             1338               68  
                             kanchugaranahalli kanchugaranahalli     3358               45  
belagavi chikkodi hukkeri    shahabandar       bassapur              1095               27  
                  chikodi    navalihal         khadaklat             3583               83  
tumakuru          gubbi      hindiskere        hindiskere            2606               37

Bottom 10 GPs (Grade 6):


,,,,,accuracy,count,n_students
District,Block,Cluster,GP Name,GP ID,,,
davanagere,jagalur,hosakere,kyasenahalli,3975,0.213559,1180,59
bidar,basavakalyan,basavaklyan urdu rural,rajeshwar,5059,0.212069,580,29
belagavi chikkodi,mudalgi,shindikurbet,durdundi,1966,0.211250,1600,80
bidar,bidar,rekulgi urdu,bagdal,921,0.203061,980,49
chamarajanagara,hanur,m.m.hills,m m hills,4037,0.201429,700,35
bidar,humnabad,mannaekheli (urdu),mannaekhelli,4266,0.200000,640,32
chitradurga,molakalmur,nagasamudra,hanagal,2382,0.186486,740,37
belagavi chikkodi,mudalgi,kaujalgi,kulagod,3859,0.166667,540,27
ballari,ballari east,urdu ballari east,sanjeevarayanakote,5210,0.156250,800,40


### Visualizing the GP-level accuracy distribution by grade

With ~5,000 reliable GPs per grade, a bar/heatmap per GP is unreadable, so we
visualize the **distribution** of GP accuracy instead:

- A **violin plot per grade** shows the full shape of the distribution
  (skew, spread, how heavy the low-performing tail is) across all GPs — much
  more informative than a single mean for thousands of units.
- An **embedded box plot** adds the median and inter-quartile range so the
  exact summary stats are visible on the same chart.
- A shared y-axis makes the grade-to-grade comparison direct.


In [27]:
# Violin (+ box) of GP-level accuracy, one violin per grade.
plot_df = gp_by_grade_reliable.reset_index()

fig = go.Figure()
for grade_val in grades:
    g = plot_df[plot_df["grade"] == grade_val]
    color = grade_colors.get(grade_val, "#AB63FA")
    fig.add_trace(
        go.Violin(
            y=g["accuracy"],
            name=f"Grade {grade_val}",
            line_color=color,
            fillcolor=color,
            opacity=0.55,
            box_visible=True,
            meanline_visible=True,
            points=False,
            hovertemplate=(
                f"Grade {grade_val}<br>"
                "Accuracy: %{y:.1%}<br>"
                f"n_GPs = {len(g):,}<extra></extra>"
            ),
        )
    )

fig.update_yaxes(
    title_text="GP mean accuracy",
    tickformat=".0%",
    gridcolor="rgba(0,0,0,0.08)",
    range=[0, 1],
)
fig.update_layout(
    title=dict(
        text=(
            "Distribution of GP-level accuracy by grade"
            f"<br><sup>Each violin = ~5,000 GPs (>= {MIN_STUDENTS_PER_GRADE} students/grade); "
            "box = median + IQR, dashed line = mean</sup>"
        ),
        x=0.02,
        xanchor="left",
    ),
    template="plotly_white",
    font=dict(size=13),
    width=850,
    height=560,
    margin=dict(t=100),
    showlegend=False,
)
fig.show()


#### How to read this chart — GP accuracy by grade

Each violin shows the full spread of ~5,000 GPs for one grade; the fatter the
violin at a given height, the more GPs sit at that accuracy level. The white
box marks the median and inter-quartile range (IQR), the dashed line the mean.

**What the data says:**

- **The three grades are almost identical.** Median GP accuracy is ~53% (Gr 4),
  ~53% (Gr 5), ~54% (Gr 6); means are all ~54%. There is **no drop-off or
  improvement across grades** — a typical GP performs about the same in Grade 4,
  5, and 6.
- **The whole system sits just above half-marks.** Even the best grade has a
  median of only ~54%, so weak performance is broad, not concentrated in one
  grade.
- **Spread narrows as grade rises.** Grade 4 is the most polarized
  (std 0.16; the middle 80% of GPs span 34% → 77%), while Grade 6 is tighter
  (std 0.13; 39% → 72%). Early-grade foundational skills are the most unevenly
  established; by Grade 6 GPs converge somewhat (partly real learning, partly
  the weakest students dropping out of later contests).
- **Grade 4 has the heaviest low tail** (min 7%) *and* the highest ceiling
  (max 97%) — the gap between the strongest and weakest GPs is widest here.

**Takeaway:** interventions should target *which GPs* struggle rather than
*which grade*, and Grade 4 is where GP-to-GP inequality is largest.


### Pooled GP-level accuracy, spread within each district

The grade-split violins above summarize each grade as a whole. To connect the
**pooled** GP-level table (all grades combined) back to geography, this
horizontal box plot shows the distribution of GP accuracy *inside* each
district, districts sorted by their median GP. Unlike the earlier district
heatmap (which only showed a single district average), this reveals the
**within-district spread** — e.g. districts with a wide box have both strong
and struggling GPs, which matters for where targeted support is needed.


In [28]:
# Horizontal box plot: pooled GP accuracy distribution per district,
# districts ordered by their median GP accuracy.
gp_pooled = gp_overall_reliable.reset_index()
district_median_order = (
    gp_pooled.groupby("District")["accuracy"].median().sort_values().index.tolist()
)

fig = go.Figure()
fig.add_trace(
    go.Box(
        x=gp_pooled["accuracy"],
        y=gp_pooled["District"],
        orientation="h",
        marker_color="#00A6A6",
        line_color="#00778B",
        boxpoints=False,
        hovertemplate="District: %{y}<br>GP accuracy: %{x:.1%}<extra></extra>",
    )
)

fig.update_xaxes(
    title_text="GP mean accuracy (pooled across grades/years)",
    tickformat=".0%",
    gridcolor="rgba(0,0,0,0.08)",
    range=[0, 1],
)
fig.update_yaxes(
    categoryorder="array",
    categoryarray=district_median_order,
    title_text="District",
)
fig.update_layout(
    title=dict(
        text=(
            "Within-district spread of GP-level accuracy"
            "<br><sup>Each box = the district's reliable GPs; sorted by median GP accuracy (low to high)</sup>"
        ),
        x=0.02,
        xanchor="left",
    ),
    template="plotly_white",
    font=dict(size=11),
    width=950,
    height=850,
    margin=dict(t=100, l=160),
)
fig.show()


#### How to read this chart — spread within each district

Each box is one district's reliable GPs, sorted top-to-bottom by median GP
accuracy. The **box width (IQR)** tells you how consistent GPs are *inside*
that district; the **box position** tells you the district's overall level.

**What the data says:**

- **Districts differ far more in level than in internal consistency.** The
  typical district's internal IQR is only ~0.15, so most of the variation is
  *between* districts (already seen in the district heatmap), not within them.
- **Two profiles emerge, giving four action groups:**

  | | Low median | High median |
  |---|---|---|
  | **Narrow spread** | Systemic problem → district-wide fix (**kalaburgi** 0.41, **raichur** 0.47, **vijayanagar** 0.42, **koppal** 0.47) | Best practice to replicate (**dakshina kannada** 0.72) |
  | **Wide spread** | Triage the worst GPs (**bidar** 0.44, IQR 0.20) | Scale the strong GPs, lift the laggards (**belagavi chikkodi** 0.64 / IQR 0.26, **bagalkot**, **vijayapura**) |

- **Wide-box districts** (belagavi chikkodi, vijayapura, bagalkot) hide both
  strong and struggling GPs under one administration — a district-average
  intervention would misfire; **GP-targeted** support is needed.
- **Narrow-and-low districts** (the Kalyana-Karnataka belt: kalaburgi, raichur,
  bidar, ballari, yadagiri, koppal) are uniformly struggling → **systemic,
  district-wide** intervention fits.
- **Narrow-and-high** dakshina kannada is a consistent-strong model to learn
  from.

**Caveat:** the tightest/highest districts (dakshina kannada n=22, uttara
kannada sirsi n=12) have very few GPs, so their "consistency" is partly small
sample — treat those conclusions cautiously.


In [35]:
from src.nfhs_district import build_contest_district_nfhs

nfhs_ctx = build_contest_district_nfhs()  # 31 rows, key: contest_district_value

# --- Student-weighted district accuracy -------------------------------------
# Each row of assessment_df is a single student sitting, so a plain mean of
# pct_correct is already weighted by the number of students per district.
district_accuracy = (
    assessment_df.groupby("District")
    .agg(accuracy=("pct_correct", "mean"), n_students=("pct_correct", "size"))
    .reset_index()
    .rename(columns={"District": "contest_district_value"})
)

district_grade_accuracy = (
    assessment_df.groupby(["District", "grade"])
    .agg(accuracy=("pct_correct", "mean"), n_students=("pct_correct", "size"))
    .reset_index()
    .rename(columns={"District": "contest_district_value"})
)

# --- Join NFHS-5 district context -------------------------------------------
enriched = district_accuracy.merge(nfhs_ctx, on="contest_district_value", how="left")
enriched_grade = district_grade_accuracy.merge(nfhs_ctx, on="contest_district_value", how="left")

print("enriched:", enriched.shape, "| enriched_grade:", enriched_grade.shape)
enriched.sort_values("accuracy")[
    ["contest_district_value", "accuracy", "n_students",
     "women_10yr_schooling_n5", "child_stunted_n5", "improved_sanitation_n5", "is_proxy"]
].head(10)


enriched: (31, 42) | enriched_grade: (93, 43)


,contest_district_value,accuracy,n_students,women_10yr_schooling_n5,child_stunted_n5,improved_sanitation_n5,is_proxy
16,kalaburgi,0.420057,91823,42.0,34.5,36.5,False
1,ballari,0.422798,46061,39.9,36.1,64.1,False
9,chitradurga,0.424103,87338,48.3,36.0,63.1,False
28,vijayanagar,0.432556,61409,39.9,36.1,64.1,True
5,bidar,0.454034,42933,45.0,36.8,56.5,False
22,raichur,0.465686,70124,31.7,39.8,53.0,False
19,koppal,0.473977,63307,34.4,49.1,58.8,False
30,yadagiri,0.482332,44853,26.4,57.6,37.4,False
7,chikkaballapura,0.486861,36469,48.0,31.3,84.9,False
11,davanagere,0.492885,38336,47.1,38.4,83.3,False


In [31]:
nfhs_ctx

,contest_district_value,standard_district,nfhs_name,is_proxy,child_adequate_diet_n5,child_marriage_n5,child_stunted_n5,child_underweight_n5,child_wasted_n5,clean_cooking_fuel_n5,...,health_insurance_delta,improved_sanitation_delta,improved_water_delta,menstrual_hygiene_delta,population_under_15_delta,sex_ratio_at_birth_delta,sex_ratio_total_delta,teen_motherhood_delta,women_10yr_schooling_delta,women_low_bmi_delta
0,bagalkot,Bagalkote,Bagalkot,False,6.1,38.7,48.3,42.3,16.9,56.6,...,-14.9,28.3,1.2,3.8,-1.7,80.0,44.0,-5.4,6.0,-4.2
1,ballari,Ballari,Bellary,False,9.7,22.2,36.1,36.5,22.9,76.2,...,4.2,24.4,-3.1,27.3,-5.5,186.0,86.0,-10.1,13.0,-1.4
2,belagavi,Belagavi,Belgaum,False,8.8,32.8,32.8,36.9,23.6,74.8,...,3.8,23.7,-4.6,5.5,-1.5,-75.0,72.0,-0.4,8.4,1.6
3,belagavi chikkodi,Belagavi,Belgaum,False,8.8,32.8,32.8,36.9,23.6,74.8,...,3.8,23.7,-4.6,5.5,-1.5,-75.0,72.0,-0.4,8.4,1.6
4,bengaluru rural,Bengaluru Rural,Bangalore Rural,False,17.6,14.1,36.6,23.8,16.2,93.6,...,-3.7,11.1,-1.7,13.7,-2.5,-136.0,-8.0,-5.6,6.7,-7.2
5,bidar,Bidar,Bidar,False,13.8,19.2,36.8,36.1,22.1,65.2,...,-3.7,28.5,0.9,21.2,1.2,-177.0,16.0,-3.6,-1.4,-1.1
6,chamarajanagara,Chamarajanagara,Chamarajanagar,False,17.8,19.3,32.2,28.7,18.0,89.0,...,7.4,38.9,1.0,29.8,-2.0,-188.0,-31.0,-2.4,11.4,-8.2
7,chikkaballapura,Chikkaballapura,Chikkaballapura,False,18.1,27.1,31.3,25.2,16.1,89.4,...,10.0,32.4,-0.5,30.1,-1.7,322.0,44.0,-1.9,8.9,-2.2
8,chikkamagaluru,Chikkamagaluru,Chikmagalur,False,15.4,19.5,27.3,25.4,24.9,79.0,...,12.3,29.3,4.6,23.0,-1.0,-595.0,-18.0,-1.9,7.7,-10.7
9,chitradurga,Chitradurga,Chitradurga,False,27.2,20.7,36.0,32.4,17.9,80.6,...,-10.9,19.7,-1.3,27.4,2.0,118.0,77.0,-0.5,1.4,-8.2


In [36]:
# --- Per-grade correlation of district accuracy with NFHS-5 context ---------
# Aggregate accuracy to the NFHS district grain (student-weighted) so each NFHS
# context row maps to a single accuracy value. This avoids pseudo-replication
# from split contest districts that share one NFHS parent (e.g. belagavi +
# belagavi chikkodi -> Belgaum; vijayanagar folds into its Bellary proxy).
_amap = assessment_df.merge(
    nfhs_ctx[["contest_district_value", "nfhs_name"]],
    left_on="District", right_on="contest_district_value", how="left",
)
nfhs_grade_acc = (
    _amap.groupby(["nfhs_name", "grade"])
    .agg(accuracy=("pct_correct", "mean"), n_students=("pct_correct", "size"))
    .reset_index()
)
nfhs_ind = nfhs_ctx.drop_duplicates("nfhs_name").set_index("nfhs_name")

# Indicators most plausibly linked to early numeracy outcomes
focus = [
    "women_10yr_schooling_n5", "female_ever_school_n5",
    "child_stunted_n5", "child_underweight_n5", "child_adequate_diet_n5",
    "improved_sanitation_n5", "clean_cooking_fuel_n5", "electricity_n5",
    "child_marriage_n5", "teen_motherhood_n5", "health_insurance_n5",
]

rows = []
for g in sorted(nfhs_grade_acc["grade"].unique()):
    sub = nfhs_grade_acc[nfhs_grade_acc["grade"] == g].merge(
        nfhs_ind[focus], left_on="nfhs_name", right_index=True
    )
    for col in focus:
        rows.append({
            "indicator": col.replace("_n5", ""),
            "grade": int(g),
            "spearman_r": round(sub["accuracy"].corr(sub[col], method="spearman"), 3),
            "n_districts": int(sub[col].notna().sum()),
        })

corr_by_grade = (
    pd.DataFrame(rows)
    .pivot(index="indicator", columns="grade", values="spearman_r")
    .sort_values(by=4, key=lambda s: s.abs(), ascending=False)
)
print("districts per grade:", nfhs_grade_acc.groupby("grade")["nfhs_name"].nunique().to_dict())
corr_by_grade


districts per grade: {4: 28, 5: 28, 6: 28}


grade,4,5,6
indicator,,,
female_ever_school,0.639,0.658,0.596
improved_sanitation,0.597,0.640,0.568
women_10yr_schooling,0.557,0.614,0.541
child_marriage,-0.419,-0.469,-0.444
health_insurance,0.386,0.467,0.450
child_stunted,-0.384,-0.389,-0.299
child_underweight,-0.283,-0.337,-0.244
teen_motherhood,-0.257,-0.259,-0.291
child_adequate_diet,-0.217,-0.146,-0.228


#### How to read `corr_by_grade` — district accuracy vs NFHS-5 context

Each number is a **Spearman rank correlation** between a district's NFHS-5
indicator and its student-weighted contest accuracy, computed separately for
Grade 4, 5, and 6 across ~28 districts. It answers: *do districts that rank
high on this indicator also rank high on accuracy?*

- **Sign** = direction: `+` the two move together, `−` they move oppositely.
- **Magnitude** (0–1): roughly ~0.1 weak, ~0.3 moderate, ~0.5+ strong.
- Rank-based, so it's robust to outlier districts; it is a **district-level
  (ecological) pattern, not a student-level one**.

**Reading across the grade columns (4 / 5 / 6):** for every indicator the three
grades are almost identical — a district's health/social profile relates to
Grade 4, 5, and 6 accuracy in nearly the same way. (The NFHS values don't vary
by grade; only the contest accuracy does.)

**Caveats to state in the write-up:**
- Correlation ≠ causation — a shared driver (e.g. district development/wealth) plausibly moves both.
- Ecological: district-level, so no inference about individual students.
- NFHS numbers are whole-district (urban + rural); the contest is rural-only → this dampens the true rural gradient.
- NFHS-5 (2019–20) predates the 2022–25 assessments.
- `electricity` / `clean_cooking_fuel` sit near 0 partly because they are near-universal (little between-district spread to correlate with).

> Keep this cell as methodology / reading guidance — the interpretive wording in the final submission should be your team's own.


In [37]:
# Visualize corr_by_grade: how each NFHS-5 district indicator correlates with
# contest accuracy, per grade. Bars right of 0 = positive (higher indicator ->
# higher accuracy); left of 0 = negative. Longer bar = stronger rank relationship.
corr_long = (
    corr_by_grade.reset_index()
    .melt(id_vars="indicator", var_name="grade", value_name="spearman_r")
)
# Strongest indicators (by |r|) end up at the top of the horizontal chart.
ind_order = corr_by_grade.index.tolist()[::-1]

fig = go.Figure()
for grade_val in grades:
    d = corr_long[corr_long["grade"] == grade_val]
    color = grade_colors.get(grade_val, "#AB63FA")
    fig.add_trace(
        go.Bar(
            y=d["indicator"],
            x=d["spearman_r"],
            orientation="h",
            name=f"Grade {grade_val}",
            marker_color=color,
            hovertemplate=f"Grade {grade_val}<br>%{{y}}<br>Spearman r = %{{x:.2f}}<extra></extra>",
        )
    )

fig.add_vline(x=0, line_width=1.5, line_color="black")
for x in (-0.5, -0.3, 0.3, 0.5):
    fig.add_vline(x=x, line_width=1, line_dash="dot", line_color="rgba(0,0,0,0.25)")

fig.update_yaxes(categoryorder="array", categoryarray=ind_order, title_text="NFHS-5 district indicator")
fig.update_xaxes(
    title_text="Spearman rank correlation with district accuracy",
    range=[-0.7, 0.7],
    gridcolor="rgba(0,0,0,0.08)",
)
fig.update_layout(
    title=dict(
        text=(
            "District accuracy vs NFHS-5 context, by grade"
            "<br><sup>Bar = Spearman r across ~28 districts; right of 0 = positive, left = negative; "
            "dotted lines at |r| = 0.3 / 0.5</sup>"
        ),
        x=0.02,
        xanchor="left",
    ),
    barmode="group",
    template="plotly_white",
    font=dict(size=12),
    width=950,
    height=650,
    margin=dict(t=110, l=180),
    legend_title="Grade",
)
fig.show()


### District-wise gap analysis — turning the correlation into policy targets

A correlation is a **single number computed across districts**, so it has no
per-district value. The actionable per-district view is the **residual (gap)**:

1. Predict each district's accuracy from its health/social context using a
   **single-predictor OLS regression**. The three strongest positive correlates
   (female literacy, improved sanitation, women with 10+ yrs schooling) are
   heavily collinear, so we collapse them into **one standardized development
   index** and regress accuracy on that. A second nutrition predictor (child
   stunting) added no unique signal once the index was in, so it is dropped for
   a cleaner, non-redundant benchmark.
2. For each district, compare **observed vs expected** accuracy:
   - **gap > 0** → the district does *better* than its context predicts →
     **resilient**: something in its schools/teaching works despite the context → study & replicate.
   - **gap < 0** → it does *worse* than its context predicts → the shortfall is
     **not** explained by health disadvantage → **school-side levers** (teaching, materials, contest prep) have the most headroom here.

Combined with a median split of context × accuracy, this gives four policy groups:

| | Low accuracy | High accuracy |
|---|---|---|
| **Weak context** | Systemic — needs health *and* education investment | Resilient — learn from these |

| **Favourable context** | School-side fix — biggest quick win | Maintain |

> Methodology / targeting scaffold only — the final policy wording should be your team's own. All gaps are district-level (ecological); NFHS is whole-district (urban+rural) while the contest is rural-only, so treat magnitudes as directional.

In [49]:
import numpy as np
from sklearn.linear_model import LinearRegression

# --- District-wise gap: observed accuracy vs a development-index benchmark -----
# The strongest positive correlates (female literacy, sanitation, women's
# schooling) are heavily collinear, so we collapse them into ONE standardized
# development index and regress accuracy on that single, interpretable driver.
# (A second nutrition predictor added no unique signal - r ~ -0.64 with the
# index - so it is dropped for a cleaner, non-redundant benchmark.)
DEV_FEATURES = [
    "female_ever_school_n5",     # r ~ 0.64
    "improved_sanitation_n5",    # r ~ 0.60
    "women_10yr_schooling_n5",   # r ~ 0.56
]

dd = enriched.dropna(subset=DEV_FEATURES + ["accuracy"]).copy()

# Development index = mean of the z-scored development features (then re-standardized).
dev_z = (dd[DEV_FEATURES] - dd[DEV_FEATURES].mean()) / dd[DEV_FEATURES].std()
dd["dev_index"] = dev_z.mean(axis=1)
dd["dev_index_z"] = (dd["dev_index"] - dd["dev_index"].mean()) / dd["dev_index"].std()

X = dd[["dev_index_z"]].to_numpy()
y = dd["accuracy"].to_numpy()

model = LinearRegression().fit(X, y)
dd["expected_accuracy"] = model.predict(X)
dd["gap"] = dd["accuracy"] - dd["expected_accuracy"]
r2 = model.score(X, y)

# 2x2 policy segmentation: context = the predicted accuracy.
ctx_med = dd["expected_accuracy"].median()
acc_med = dd["accuracy"].median()


def _segment(r):
    hi_ctx, hi_acc = r["expected_accuracy"] >= ctx_med, r["accuracy"] >= acc_med
    if hi_ctx and hi_acc:
        return "Favourable context, high accuracy - maintain"
    if hi_ctx and not hi_acc:
        return "Favourable context, low accuracy - school-side fix"
    if not hi_ctx and hi_acc:
        return "Weak context, high accuracy - resilient, learn from"
    return "Weak context, low accuracy - systemic (health + education)"


dd["policy_segment"] = dd.apply(_segment, axis=1)

district_gap = (
    dd[["contest_district_value", "accuracy", "expected_accuracy", "gap",
        "dev_index_z", *DEV_FEATURES, "policy_segment", "is_proxy"]]
    .sort_values("gap")
    .reset_index(drop=True)
)

print(f"OLS on {len(dd)} districts | 1 predictor (development index) | R^2 = {r2:.3f}")
print(f"Standardized coefficient (accuracy change per +1 SD of dev index): {model.coef_[0]:+.4f}")
print(f"\nmedians -> expected accuracy = {ctx_med:.1%} | observed accuracy = {acc_med:.1%}")
print("\nPolicy segment counts:")
print(district_gap["policy_segment"].value_counts().to_string())

# Actionable shortlists.
print("\nMost UNDER-performing vs context (school-side headroom):")
print(district_gap.head(5)[["contest_district_value", "accuracy", "expected_accuracy", "gap"]]
      .assign(gap=lambda d: (d["gap"] * 100).round(1)).to_string(index=False))
print("\nMost OVER-performing vs context (resilient - study & replicate):")
print(district_gap.tail(5)[["contest_district_value", "accuracy", "expected_accuracy", "gap"]]
      .assign(gap=lambda d: (d["gap"] * 100).round(1)).to_string(index=False))
print("\n'Favourable context, low accuracy - school-side fix' districts:")
print(sorted(district_gap.loc[district_gap["policy_segment"].str.startswith("Favourable context, low"),
                              "contest_district_value"]))

# Readable copy (accuracy/gap as %), sorted worst gap first.
district_gap_display = district_gap.assign(
    accuracy=(district_gap["accuracy"] * 100).round(1),
    expected_accuracy=(district_gap["expected_accuracy"] * 100).round(1),
    gap=(district_gap["gap"] * 100).round(1),
).rename(columns={
    "accuracy": "accuracy_%", "expected_accuracy": "expected_%", "gap": "gap_pp",
})
district_gap_display


OLS on 31 districts | 1 predictor (development index) | R^2 = 0.427
Standardized coefficient (accuracy change per +1 SD of dev index): +0.0585

medians -> expected accuracy = 55.7% | observed accuracy = 56.6%

Policy segment counts:
policy_segment
Favourable context, high accuracy - maintain                  11
Weak context, low accuracy - systemic (health + education)    10
Favourable context, low accuracy - school-side fix             5
Weak context, high accuracy - resilient, learn from            5

Most UNDER-performing vs context (school-side headroom):
contest_district_value  accuracy  expected_accuracy   gap
           chitradurga  0.424103           0.552601 -12.8
                kodagu  0.543866           0.645318 -10.1
            davanagere  0.492885           0.579052  -8.6
               ballari  0.422798           0.508071  -8.5
           vijayanagar  0.432556           0.508071  -7.6

Most OVER-performing vs context (resilient - study & replicate):
contest_district_val

,contest_district_value,accuracy_%,expected_%,gap_pp,dev_index_z,female_ever_school_n5,improved_sanitation_n5,women_10yr_schooling_n5,policy_segment,is_proxy
0,chitradurga,42.4,55.3,-12.8,-0.053962,73.0,63.1,48.3,"Weak context, low accuracy - systemic (health ...",False
1,kodagu,54.4,64.5,-10.1,1.531244,83.6,93.9,58.9,"Favourable context, low accuracy - school-side...",False
2,davanagere,49.3,57.9,-8.6,0.398276,74.9,83.3,47.1,"Favourable context, low accuracy - school-side...",False
3,ballari,42.3,50.8,-8.5,-0.815286,64.2,64.1,39.9,"Weak context, low accuracy - systemic (health ...",False
4,vijayanagar,43.3,50.8,-7.6,-0.815286,64.2,64.1,39.9,"Weak context, low accuracy - systemic (health ...",True
5,kolar,52.0,59.6,-7.5,0.685027,71.6,89.2,55.0,"Favourable context, low accuracy - school-side...",False
6,chikkaballapura,48.7,55.7,-7.0,0.026752,65.9,84.9,48.0,"Favourable context, low accuracy - school-side...",False
7,bidar,45.4,52.3,-6.9,-0.560519,68.2,56.5,45.0,"Weak context, low accuracy - systemic (health ...",False
8,kalaburgi,42.0,48.3,-6.3,-1.247449,65.0,36.5,42.0,"Weak context, low accuracy - systemic (health ...",False
9,tumakuru madhugiri,56.6,60.4,-3.7,0.819069,72.3,86.1,58.9,"Favourable context, high accuracy - maintain",False


In [50]:
# Scatter: observed vs benchmark (multi-feature predicted) accuracy.
# The dashed 45-degree line is "on benchmark" (gap = 0); points above it
# over-perform their health/social context, points below under-perform.
lo = float(min(dd["expected_accuracy"].min(), dd["accuracy"].min())) - 0.02
hi = float(max(dd["expected_accuracy"].max(), dd["accuracy"].max())) + 0.02

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi], mode="lines",
    line=dict(color="black", dash="dash", width=2),
    name="On benchmark (gap = 0)", hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=dd["expected_accuracy"], y=dd["accuracy"], mode="markers+text",
    text=dd["contest_district_value"], textposition="top center",
    textfont=dict(size=9),
    marker=dict(
        size=12, color=dd["gap"], colorscale="RdYlGn", cmid=0,
        line=dict(color="rgba(0,0,0,0.4)", width=1),
        colorbar=dict(title="Gap", tickformat="+.0%"),
    ),
    customdata=dd["gap"].to_numpy().reshape(-1, 1),
    hovertemplate=(
        "%{text}<br>Expected: %{x:.1%}<br>Observed: %{y:.1%}"
        "<br>Gap: %{customdata[0]:+.1%}<extra></extra>"
    ),
    name="District",
))
fig.update_xaxes(
    title_text="Expected accuracy (development-index benchmark)",
    tickformat=".0%", range=[lo, hi], gridcolor="rgba(0,0,0,0.08)",
)
fig.update_yaxes(
    title_text="Observed contest accuracy",
    tickformat=".0%", range=[lo, hi], gridcolor="rgba(0,0,0,0.08)",
)
fig.update_layout(
    title=dict(
        text=(
            "Observed vs expected district accuracy (development-index benchmark)"
            "<br><sup>Dashed 45deg line = on benchmark; above = over-performs context, below = under-performs</sup>"
        ),
        x=0.02, xanchor="left",
    ),
    template="plotly_white", font=dict(size=12), width=850, height=720,
    margin=dict(t=100), showlegend=False,
)
fig.show()


In [51]:
# Actionable ranking: how far each district sits above/below its context
# benchmark. Red (left of 0) = under-performs -> school-side intervention has
# the most headroom; green (right) = resilient models to learn from.
gap_sorted = district_gap.sort_values("gap")
bar_colors = ["#2ca02c" if v >= 0 else "#d62728" for v in gap_sorted["gap"]]
bar_labels = [
    d + (" *" if proxy else "")
    for d, proxy in zip(gap_sorted["contest_district_value"], gap_sorted["is_proxy"])
]

fig = go.Figure(go.Bar(
    y=bar_labels, x=gap_sorted["gap"], orientation="h",
    marker_color=bar_colors,
    customdata=gap_sorted[["accuracy", "expected_accuracy"]].values,
    hovertemplate=(
        "%{y}<br>Gap = %{x:+.1%}<br>Accuracy: %{customdata[0]:.1%}"
        "<br>Expected: %{customdata[1]:.1%}<extra></extra>"
    ),
))
fig.add_vline(x=0, line_width=1.5, line_color="black")
fig.update_xaxes(
    title_text="Accuracy gap vs health-context benchmark (residual)",
    tickformat="+.0%", gridcolor="rgba(0,0,0,0.08)",
)
fig.update_yaxes(title_text="District")
fig.update_layout(
    title=dict(
        text=(
            "Which districts beat or miss their health-context benchmark?"
            "<br><sup>Residual from a development-index OLS benchmark; red = under-performs (school-side headroom), "
            "green = resilient. * = NFHS proxy (vijayanagar -> Bellary)</sup>"
        ),
        x=0.02, xanchor="left",
    ),
    template="plotly_white", font=dict(size=11), width=950, height=760,
    margin=dict(t=110, l=170), showlegend=False,
)
fig.show()
